In [1]:
import json
import clickhouse_connect
import pandas as pd

# Khởi tạo kết nối tới ClickHouse Server
client = clickhouse_connect.get_client(
    host="applog.xomdata.com",
    port=80,
    username="xomdata",
    password="Vyljk8uhGfkR25vuocmNaJ1wJjtJ6920EANi0JkU",
)

TARGET_DB = "laplaptech"
print(f"✅ Kết nối thành công tới ClickHouse Server version: {client.server_version}")

✅ Kết nối thành công tới ClickHouse Server version: 26.7.1.1315


In [2]:
import json
import pandas as pd

def inspect_json_structure(val, prefix=""):
    """Hàm đệ quy quét cấu trúc và kiểu dữ liệu của các key bên trong JSON."""
    keys = []
    if isinstance(val, dict):
        for k, v in val.items():
            full_path = f"{prefix}.{k}" if prefix else str(k)
            if isinstance(v, dict):
                keys.extend(inspect_json_structure(v, full_path))
            elif isinstance(v, list):
                if len(v) > 0 and isinstance(v[0], dict):
                    keys.extend(inspect_json_structure(v[0], f"{full_path}[]"))
                else:
                    elem_type = (
                        type(v[0]).__name__ if len(v) > 0 else "unknown"
                    )
                    keys.append(f"{full_path}: Array[{elem_type}]")
            else:
                keys.append(f"{full_path}: {type(v).__name__}")
    return keys

# Danh sách các bảng cần thiết từ laplaptech_schema.md
TARGET_TABLES = [
    "brand",
    "cpu_model",
    "gpu_model",
    "laptop_model",
    "laptop_benchmark_result",
    "user_event_tracking"
]

# 1. Lấy metadata danh sách các cột từ system.columns
metadata_query = f"""
SELECT 
    table,
    name AS column_name,
    type AS data_type
FROM system.columns
WHERE database = '{TARGET_DB}' AND table IN {tuple(TARGET_TABLES)}
ORDER BY table, position
"""
df_meta = client.query_df(metadata_query)

all_table_schemas = []

# 2. Quét từng bảng an toàn với try...except
for table_name, group in df_meta.groupby("table"):
    sample_df = pd.DataFrame()

    # Bọc try-catch khi lấy sample: Nếu bảng/view bị lỗi metadata, không làm dừng tiến trình
    try:
        sample_df = client.query_df(
            f"SELECT * FROM `{TARGET_DB}`.`{table_name}` LIMIT 1"
        )
    except Exception as e:
        # Bảng trống, view hỏng hoặc không có quyền đọc -> tiếp tục lấy schema tĩnh từ metadata
        pass

    for _, row in group.iterrows():
        col_name = row["column_name"]
        col_type = row["data_type"]
        json_fields = []

        if not sample_df.empty and col_name in sample_df.columns:
            sample_val = sample_df[col_name].iloc[0]

            # TH1: Native Object / Dict / Map của ClickHouse
            if isinstance(sample_val, dict):
                json_fields = inspect_json_structure(sample_val)

            # TH2: Cột dạng String nhưng chứa chuỗi JSON
            elif isinstance(sample_val, str):
                cleaned_val = sample_val.strip()
                if (
                    cleaned_val.startswith("{") and cleaned_val.endswith("}")
                ) or (
                    cleaned_val.startswith("[") and cleaned_val.endswith("]")
                ):
                    try:
                        parsed_json = json.loads(cleaned_val)
                        json_fields = inspect_json_structure(parsed_json)
                    except Exception:
                        pass

        all_table_schemas.append(
            {
                "Table": table_name,
                "Column Name": col_name,
                "Data Type": col_type,
                "Is JSON": "Yes" if len(json_fields) > 0 else "No",
                "JSON Sub-fields / Schema": (
                    "\n".join(json_fields) if json_fields else "-"
                ),
            }
        )

# 3. Xuất kết quả
df_final_schema = pd.DataFrame(all_table_schemas)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

print(f"✅ Hoàn tất phân tích schema database '{TARGET_DB}':")
display(df_final_schema)


✅ Hoàn tất phân tích schema database 'laplaptech':


,Table,Column Name,Data Type,Is JSON,JSON Sub-fields / Schema
0,brand,created_on,Nullable(DateTime64(3)),No,-
1,brand,changed_on,Nullable(DateTime64(3)),No,-
2,brand,id,Nullable(Int64),No,-
3,brand,name,Nullable(String),No,-
4,brand,created_by_fk,Nullable(Int64),No,-
5,brand,changed_by_fk,Nullable(Int64),No,-
6,brand,is_chip_brand,Nullable(Bool),No,-
7,brand,elton_created_at,Nullable(DateTime64(3)),No,-
8,cpu_model,created_on,Nullable(DateTime64(3)),No,-
9,cpu_model,changed_on,Nullable(DateTime64(3)),No,-


# EDA

### Sample data table `brand`

In [3]:
# Gợi ý: In ra 10 dòng dữ liệu đầu tiên của bảng brand
df_brand = client.query_df(f"SELECT * FROM `{TARGET_DB}`.`brand` LIMIT 10")
display(df_brand)


,created_on,changed_on,id,name,created_by_fk,changed_by_fk,is_chip_brand,elton_created_at
0,2024-12-20 22:30:44.284,2024-12-20 22:30:44.284,1,Asus,1,1,None,2026-08-03 20:23:11.738
1,2024-12-20 22:30:45.759,2024-12-20 22:30:54.195,2,Acer,1,1,None,2026-08-03 20:23:11.738
2,2024-12-21 19:17:36.774,2024-12-21 19:17:36.774,3,Lenovo,2,2,None,2026-08-03 20:23:11.738
3,2024-12-21 19:17:41.198,2024-12-21 19:17:41.198,4,Dell,2,2,None,2026-08-03 20:23:11.738
4,2024-12-21 19:17:45.242,2024-12-21 19:17:45.242,5,HP,2,2,None,2026-08-03 20:23:11.738
5,2024-12-27 09:10:55.256,2024-12-29 10:41:55.873,6,Apple,1,1,True,2026-08-03 20:23:11.738
6,2024-12-27 10:04:16.429,2024-12-27 10:04:16.429,7,MSI,1,1,None,2026-08-03 20:23:11.738
7,2024-12-28 22:03:32.879,2024-12-28 22:03:32.879,8,Microsoft,2,2,None,2026-08-03 20:23:11.738
8,2024-12-29 08:30:06.810,2024-12-29 08:30:06.810,9,Gigabyte,2,2,None,2026-08-03 20:23:11.738
9,2024-12-29 10:41:45.521,2024-12-29 10:41:45.521,10,Intel,1,1,True,2026-08-03 20:23:11.738


### Sample data table `cpu_model`

In [4]:
# Gợi ý: In ra 10 dòng dữ liệu đầu tiên của bảng cpu_model
df_cpu_model = client.query_df(f"SELECT * FROM `{TARGET_DB}`.`cpu_model` LIMIT 10")
display(df_cpu_model)


,created_on,changed_on,id,name,created_by_fk,changed_by_fk,brand_id,is_active,elton_created_at
0,2024-12-21 19:16:31.921,2024-12-21 19:16:31.921,1,Intel Core Ultra 7 258V,1,1,10,True,2026-08-03 20:23:12.343
1,2024-12-21 19:16:40.148,2024-12-29 12:30:59.205,2,Intel Core Ultra 7 256V,1,2,10,False,2026-08-03 20:23:12.343
2,2024-12-21 19:16:48.146,2024-12-21 19:16:48.146,3,Apple M4,1,1,6,True,2026-08-03 20:23:12.343
3,2024-12-21 19:16:52.396,2024-12-21 19:16:52.396,4,Apple M4 Pro,1,1,6,True,2026-08-03 20:23:12.343
4,2024-12-21 19:16:56.235,2024-12-21 19:16:56.235,5,Apple M4 Max,1,1,6,True,2026-08-03 20:23:12.343
5,2024-12-21 19:17:12.216,2024-12-21 19:17:12.216,6,AMD Ryzen AI 9 HX 370,1,1,11,True,2026-08-03 20:23:12.343
6,2024-12-21 19:18:09.310,2024-12-21 19:18:17.757,7,AMD Ryzen AI 9 365,2,2,11,True,2026-08-03 20:23:12.343
7,2024-12-27 09:02:01.693,2024-12-27 09:02:01.693,8,Intel Core Ultra 7 155H,1,1,10,True,2026-08-03 20:23:12.343
8,2024-12-27 09:36:16.526,2024-12-27 09:36:16.526,9,Intel Core Ultra 9 185H,1,1,10,True,2026-08-03 20:23:12.343
9,2024-12-27 09:36:29.901,2024-12-27 09:36:29.901,10,Intel Core i7-13650HX,1,1,10,True,2026-08-03 20:23:12.343


### Sample data table `gpu_model`

In [5]:
# Gợi ý: In ra 10 dòng dữ liệu đầu tiên của bảng gpu_model
df_gpu_model = client.query_df(f"SELECT * FROM `{TARGET_DB}`.`gpu_model` LIMIT 10")
display(df_gpu_model)


,created_on,changed_on,id,name,created_by_fk,changed_by_fk,brand_id,is_active,elton_created_at
0,2024-12-21 19:17:22.568,2025-01-18 15:16:51.260,1,NVIDIA RTX 4070,1,1,13,True,2026-08-03 20:23:11.706
1,2024-12-21 19:17:27.387,2025-01-18 15:16:57.518,2,NVIDIA RTX 4060,1,1,13,True,2026-08-03 20:23:11.706
2,2024-12-21 19:17:57.902,2025-01-18 15:17:05.910,3,Intel Arc Graphics 140V (tích hợp),1,1,10,True,2026-08-03 20:23:11.706
3,2024-12-27 09:23:24.516,2025-01-18 15:17:22.840,4,Apple M4,1,1,6,True,2026-08-03 20:23:11.706
4,2024-12-27 09:23:30.697,2025-01-18 15:17:12.638,5,Apple M4 Pro,1,1,6,True,2026-08-03 20:23:11.706
5,2024-12-27 09:23:40.036,2025-01-18 15:17:26.213,6,Apple M4 Max,1,1,6,True,2026-08-03 20:23:11.706
6,2024-12-28 21:24:20.718,2025-01-18 15:16:28.321,7,NVIDIA RTX 4050,2,1,13,True,2026-08-03 20:23:11.706
7,2024-12-28 21:24:42.663,2025-01-18 15:17:29.320,8,NVIDIA RTX 4080,2,1,13,True,2026-08-03 20:23:11.706
8,2024-12-28 21:30:55.154,2025-01-18 15:16:32.349,9,NVIDIA RTX 3050,2,1,13,True,2026-08-03 20:23:11.706
9,2024-12-28 21:34:17.162,2025-01-18 15:16:35.891,10,NVIDIA RTX 2050,2,1,13,True,2026-08-03 20:23:11.706


### Sample data table `laptop_model`

In [6]:
# Gợi ý: In ra 10 dòng dữ liệu đầu tiên của bảng laptop_model
df_laptop_model = client.query_df(f"SELECT * FROM `{TARGET_DB}`.`laptop_model` LIMIT 10")
display(df_laptop_model)


,created_on,changed_on,id,name,is_gaming_laptop,year_introduce,cpu_note,cpu_tdp,gpu_note,battery_capacity_whr,...,screen_dimension_width,screen_dimension_height,screen_ppi,laptop_weight,charger_weight,brand_model_codename,thumbnail_image_url,is_workstation,is_mobile_device,elton_created_at
0,2024-12-21 19:19:05.122,2025-07-28 08:12:23.594,1,Asus Zenbook S 14 OLED,False,2024,<NA>,28,<NA>,72.0,...,2880.0,1800.0,243.00,1200.0,NaN,UX5406,8fa37118-1686-11f0-a4f6-9ad51ddfc6b6_sep_asus_zenbook_s_14_ux5406_white_1_ae1d1cf027.png,False,None,2026-08-03 20:23:11.979
1,2024-12-21 20:10:09.559,2025-01-13 10:15:18.385,2,Expertbook P5,False,2024,<NA>,Tiêu thụ trung bình 31W,<NA>,63.0,...,2560.0,1600.0,215.63,1269.0,325.0,ASUS P5405CSA-NZ0017W,9cccf1c6-d15c-11ef-b650-5e7ddee1ac02_sep_p5.png,None,None,2026-08-03 20:23:11.979
2,2024-12-23 11:29:11.153,2025-01-19 13:00:28.937,3,Dell XPS 13 2024,False,2024,<NA>,<NA>,<NA>,55.0,...,1920.0,1200.0,174.00,1220.0,NaN,<NA>,2a8878e4-d0e2-11ef-b3f0-9ac5700f99a9_sep_xps-13-oled-graphite-2.png,False,None,2026-08-03 20:23:11.979
3,2024-12-27 09:05:00.964,2025-01-13 19:24:37.477,4,HP OMEN Transcend 14,True,2024,<NA>,<NA>,<NA>,71.0,...,2880.0,1800.0,243.00,1640.0,496.0,<NA>,5a25e4a2-d1a9-11ef-9138-0ebcfd24c896_sep_laptop-hp-omen-transcend-14-2024-white-ultra-9-185h-rtx-4070-ram-32gb-ssd-2tb-14-2-8k-120hz-oled-7.png,None,None,2026-08-03 20:23:11.979
4,2024-12-27 09:07:11.266,2025-01-15 17:00:29.200,5,Asus TUF Gaming A14 (2024),True,2024,Tiêu thụ điện trung bình 80W,<NA>,<NA>,72.4,...,2560.0,1600.0,215.63,1400.0,585.0,<NA>,8b495af8-d327-11ef-a09b-be31f04d3140_sep_8ef79421-e9c6-4c8b-8529-0e0dc2a09952.png,None,None,2026-08-03 20:23:11.979
5,2024-12-27 09:12:50.189,2025-04-13 17:01:05.639,6,MacBook Pro 14 inch M4 Pro (base),False,2024,<NA>,<NA>,<NA>,72.4,...,3024.0,1964.0,258.00,1600.0,NaN,<NA>,ed584792-d0af-11ef-8bc9-26a18b537b9a_sep_0031844_den_550.jpg,False,None,2026-08-03 20:23:11.979
6,2024-12-27 09:25:14.505,2025-01-12 13:39:33.988,7,MacBook Pro 16 inch M4 Max,False,2024,<NA>,<NA>,<NA>,100.0,...,3456.0,2234.0,257.00,2150.0,NaN,<NA>,fb3989f2-d0af-11ef-9522-26a18b537b9a_sep_Macbook-Pro-2024-M4.jpg,None,None,2026-08-03 20:23:11.979
7,2024-12-27 09:26:15.702,2025-01-15 09:50:21.795,8,Apple MacBook Pro M4,False,2024,<NA>,<NA>,<NA>,72.4,...,3024.0,1964.0,253.93,1527.0,238.0,<NA>,75db6c7e-d2eb-11ef-9286-02286e92b121_sep_macbook_pro_m4.png,None,None,2026-08-03 20:23:11.979
8,2024-12-27 09:35:37.480,2025-04-11 10:38:35.471,9,Asus Vivobook S 14,False,2024,Tiêu thụ trung bình 36W điện,<NA>,<NA>,75.0,...,2880.0,1880.0,245.66,1291.0,220.0,S5406SA-PP059WS,71704a36-1686-11f0-b347-9ad51ddfc6b6_sep_asus_vivobook_s_14_oled_s5406_blue_1_176c88102b.png,False,None,2026-08-03 20:23:11.979
9,2024-12-28 21:29:35.471,2025-01-12 15:26:05.905,10,MSI Stealth 18 Mercedes AMG,True,2024,Tiêu thụ trung bình 110W,<NA>,<NA>,99.0,...,3840.0,2400.0,251.57,2924.0,772.0,A1VHG 080VN,dd655a1e-d0be-11ef-bc3f-9ac5700f99a9_sep_IMG_2880.webp,None,None,2026-08-03 20:23:11.979


### Sample data table `laptop_benchmark_result`

In [7]:
# Gợi ý: In ra 10 dòng dữ liệu đầu tiên của bảng laptop_benchmark_result
df_laptop_benchmark_result = client.query_df(f"SELECT * FROM `{TARGET_DB}`.`laptop_benchmark_result` LIMIT 10")
display(df_laptop_benchmark_result)


,created_on,changed_on,id,office_battery_result_minutes,gaming_battery_result_minutes,note,laptop_model_id,created_by_fk,changed_by_fk,geekbench_6_compute_gpu_plugged_in,geekbench_6_compute_gpu_battery,is_active,geekbench_6_cpu_single_core_plugged_in,geekbench_6_cpu_single_core_battery,geekbench_6_cpu_multi_core_plugged_in,geekbench_6_cpu_multi_core_battery,review_video_url,foldable_opening_battery_result_minutes,elton_created_at
0,2024-12-21 19:19:44.699,2025-02-21 05:10:31.873,1,933.0,NaN,<NA>,1,1,1,29164.0,22935.0,True,2721.0,1742.0,10729.0,7010.0,<NA>,NaN,2026-08-03 20:23:12.113
1,2024-12-23 11:30:26.421,2024-12-23 11:30:26.421,2,844.0,NaN,<NA>,3,1,1,NaN,NaN,True,NaN,NaN,NaN,NaN,<NA>,NaN,2026-08-03 20:23:12.113
2,2024-12-27 09:08:12.514,2025-08-03 10:39:52.057,3,436.0,NaN,Máy để chế độ Silent khi rút sạc,5,1,1,93537.0,74741.0,True,2869.0,2619.0,15479.0,11893.0,https://www.youtube.com/watch?v=aeOZZEA1O0g,NaN,2026-08-03 20:23:12.113
3,2024-12-27 09:10:24.415,2024-12-27 09:10:24.415,4,789.0,NaN,<NA>,2,1,1,NaN,NaN,True,NaN,NaN,NaN,NaN,<NA>,NaN,2026-08-03 20:23:12.113
4,2024-12-27 09:24:03.336,2025-04-13 17:15:03.045,5,634.0,NaN,<NA>,6,1,1,60886.0,60585.0,True,3789.0,3825.0,20388.0,20453.0,https://www.youtube.com/watch?v=nX_3eVhjN-w,NaN,2026-08-03 20:23:12.113
5,2024-12-27 09:26:30.311,2025-04-13 17:08:53.267,6,680.0,NaN,<NA>,8,1,1,37937.0,NaN,True,3683.0,NaN,15054.0,NaN,<NA>,NaN,2026-08-03 20:23:12.113
6,2024-12-27 09:35:59.093,2024-12-29 09:02:16.869,7,770.0,NaN,<NA>,9,1,2,28764.0,NaN,True,NaN,NaN,NaN,NaN,<NA>,NaN,2026-08-03 20:23:12.113
7,2024-12-27 14:42:58.459,2024-12-27 14:42:58.459,8,339.0,NaN,<NA>,4,1,1,NaN,NaN,True,NaN,NaN,NaN,NaN,<NA>,NaN,2026-08-03 20:23:12.113
8,2024-12-28 22:14:13.640,2024-12-28 22:14:13.640,9,179.0,97.0,<NA>,10,2,2,156177.0,NaN,True,NaN,NaN,NaN,NaN,<NA>,NaN,2026-08-03 20:23:12.113
9,2024-12-28 22:16:01.178,2024-12-28 22:16:01.178,10,334.0,75.0,<NA>,11,2,2,56515.0,NaN,True,NaN,NaN,NaN,NaN,<NA>,NaN,2026-08-03 20:23:12.113


### Sample data table `user_event_tracking`

In [8]:
# Gợi ý: In ra 10 dòng dữ liệu đầu tiên của bảng user_event_tracking
df_user_event_tracking = client.query_df(f"SELECT * FROM `{TARGET_DB}`.`user_event_tracking` LIMIT 10")
display(df_user_event_tracking)


,id,event_name,user_id,event_data,device,event_local_timestamp,event_received_on_server_timestamp,session_id,user_psuedo_id,app_version,elton_created_at
0,1,pageview,3,"{""page_name"": ""DeviceDetail"", ""device_id"": ""55"", ""url"": ""https://laplap.tech/device/55"", ""referrer"": ""https://laplap.tech/""}","{""user_agent"": ""Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"", ""os_name"": ""Mac OS"", ""os_version"": ""10.15.7"", ""device_brand"": null, ""device_name"": null, ""device_type"": null, ""manufacturer"": null, ""model_id"": null, ""model_name"": null}",1736877740,1736879550,2ebb1872-5046-43bc-aba2-ea8cb37a211f,5f034237c69bd68713699694acd92124,web,2026-08-03 20:23:37.672
1,2,pageview,3,"{""page_name"": ""Home"", ""url"": ""https://laplap.tech/"", ""referrer"": ""https://laplap.tech/""}","{""user_agent"": ""Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"", ""os_name"": ""Mac OS"", ""os_version"": ""10.15.7"", ""device_brand"": null, ""device_name"": null, ""device_type"": null, ""manufacturer"": null, ""model_id"": null, ""model_name"": null}",1736877766,1736879550,2ebb1872-5046-43bc-aba2-ea8cb37a211f,5f034237c69bd68713699694acd92124,web,2026-08-03 20:23:37.672
2,3,pageview,3,"{""page_name"": ""DeviceDetail"", ""device_id"": ""54"", ""url"": ""https://laplap.tech/device/54"", ""referrer"": ""https://laplap.tech/""}","{""user_agent"": ""Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"", ""os_name"": ""Mac OS"", ""os_version"": ""10.15.7"", ""device_brand"": null, ""device_name"": null, ""device_type"": null, ""manufacturer"": null, ""model_id"": null, ""model_name"": null}",1736877768,1736879550,2ebb1872-5046-43bc-aba2-ea8cb37a211f,5f034237c69bd68713699694acd92124,web,2026-08-03 20:23:37.672
3,4,pageview,3,"{""page_name"": ""Home"", ""url"": ""https://laplap.tech/"", ""referrer"": ""https://laplap.tech/""}","{""user_agent"": ""Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"", ""os_name"": ""Mac OS"", ""os_version"": ""10.15.7"", ""device_brand"": null, ""device_name"": null, ""device_type"": null, ""manufacturer"": null, ""model_id"": null, ""model_name"": null}",1736877769,1736879550,2ebb1872-5046-43bc-aba2-ea8cb37a211f,5f034237c69bd68713699694acd92124,web,2026-08-03 20:23:37.672
4,5,pageview,3,"{""page_name"": ""DeviceDetail"", ""device_id"": ""49"", ""url"": ""https://laplap.tech/device/49"", ""referrer"": ""https://laplap.tech/""}","{""user_agent"": ""Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"", ""os_name"": ""Mac OS"", ""os_version"": ""10.15.7"", ""device_brand"": null, ""device_name"": null, ""device_type"": null, ""manufacturer"": null, ""model_id"": null, ""model_name"": null}",1736877770,1736879550,2ebb1872-5046-43bc-aba2-ea8cb37a211f,5f034237c69bd68713699694acd92124,web,2026-08-03 20:23:37.672
5,6,pageview,3,"{""page_name"": ""Home"", ""url"": ""https://laplap.tech/"", ""referrer"": ""https://laplap.tech/""}","{""user_agent"": ""Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"", ""os_name"": ""Mac OS"", ""os_version"": ""10.15.7"", ""device_brand"": null, ""device_name"": null, ""device_type"": null, ""manufacturer"": null, ""model_id"": null, ""model_name"": null}",1736877771,1736879550,2ebb1872-5046-43bc-aba2-ea8cb37a211f,5f034237c69bd68713699694acd92124,web,2026-08-03 20:23:37.672
6,7,pageview,3,"{""page_name"": ""DeviceDetail"", ""device_id"": ""47"", ""url"": ""https://laplap.tech/device/47"", ""referrer"": ""https://laplap.tech/""}","{""user_agent"": ""Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"", ""os_nam

## Phân tích & Gộp dữ liệu Laptop Performance (Laptop Model + Benchmark + CPU + GPU)

In [9]:
# SQL join các bảng liên quan đến hiệu năng laptop để tạo thành 1 bảng duy nhất với tên field gốc
query = f"""
WITH latest_benchmark AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY laptop_model_id ORDER BY changed_on DESC, id DESC) as rn
        FROM `{TARGET_DB}`.`laptop_benchmark_result`
        WHERE is_active = 1
    )
    WHERE rn = 1
)
SELECT 
    # Laptop Model original fields
    lm.id,
    lm.created_on,
    lm.changed_on,
    lm.name,
    lm.is_gaming_laptop,
    lm.year_introduce,
    lm.cpu_note,
    lm.cpu_tdp,
    lm.gpu_note,
    lm.battery_capacity_whr,
    lm.created_by_fk,
    lm.changed_by_fk,
    lm.brand_id,
    lm.cpu_model_id,
    lm.gpu_model_id,
    lm.is_visible,
    lm.is_active,
    lm.gpu_tdp,
    lm.screen_size,
    lm.screen_dimension_width,
    lm.screen_dimension_height,
    lm.screen_ppi,
    lm.laptop_weight,
    lm.charger_weight,
    lm.brand_model_codename,
    lm.thumbnail_image_url,
    lm.is_workstation,
    lm.is_mobile_device,
    lm.elton_created_at,
    
    # Benchmark fields (giữ nguyên tên gốc, tránh trùng tên với laptop_model)
    bench.office_battery_result_minutes,
    bench.gaming_battery_result_minutes,
    bench.note,
    bench.geekbench_6_compute_gpu_plugged_in,
    bench.geekbench_6_compute_gpu_battery,
    bench.geekbench_6_cpu_single_core_plugged_in,
    bench.geekbench_6_cpu_single_core_battery,
    bench.geekbench_6_cpu_multi_core_plugged_in,
    bench.geekbench_6_cpu_multi_core_battery,
    bench.review_video_url,
    bench.foldable_opening_battery_result_minutes
FROM `{TARGET_DB}`.`laptop_model` lm
LEFT JOIN latest_benchmark bench ON lm.id = bench.laptop_model_id
WHERE lm.is_active = 1
LIMIT 10
"""

df_joined = client.query_df(query)
display(df_joined)


,id,created_on,changed_on,name,is_gaming_laptop,year_introduce,cpu_note,cpu_tdp,gpu_note,battery_capacity_whr,...,gaming_battery_result_minutes,note,geekbench_6_compute_gpu_plugged_in,geekbench_6_compute_gpu_battery,geekbench_6_cpu_single_core_plugged_in,geekbench_6_cpu_single_core_battery,geekbench_6_cpu_multi_core_plugged_in,geekbench_6_cpu_multi_core_battery,review_video_url,foldable_opening_battery_result_minutes
0,1,2024-12-21 19:19:05.122,2025-07-28 08:12:23.594,Asus Zenbook S 14 OLED,False,2024,<NA>,28,<NA>,72.0,...,NaN,<NA>,29164.0,22935.0,2721.0,1742.0,10729.0,7010.0,<NA>,NaN
1,2,2024-12-21 20:10:09.559,2025-01-13 10:15:18.385,Expertbook P5,False,2024,<NA>,Tiêu thụ trung bình 31W,<NA>,63.0,...,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
2,3,2024-12-23 11:29:11.153,2025-01-19 13:00:28.937,Dell XPS 13 2024,False,2024,<NA>,<NA>,<NA>,55.0,...,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
3,4,2024-12-27 09:05:00.964,2025-01-13 19:24:37.477,HP OMEN Transcend 14,True,2024,<NA>,<NA>,<NA>,71.0,...,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
4,5,2024-12-27 09:07:11.266,2025-01-15 17:00:29.200,Asus TUF Gaming A14 (2024),True,2024,Tiêu thụ điện trung bình 80W,<NA>,<NA>,72.4,...,NaN,"Khi rút sạc, Amoury Crate chuyển về Performance Mode",94605.0,78697.0,2791.0,2807.0,14523.0,13830.0,<NA>,NaN
5,6,2024-12-27 09:12:50.189,2025-04-13 17:01:05.639,MacBook Pro 14 inch M4 Pro (base),False,2024,<NA>,<NA>,<NA>,72.4,...,NaN,<NA>,60886.0,60585.0,3789.0,3825.0,20388.0,20453.0,https://www.youtube.com/watch?v=nX_3eVhjN-w,NaN
6,7,2024-12-27 09:25:14.505,2025-01-12 13:39:33.988,MacBook Pro 16 inch M4 Max,False,2024,<NA>,<NA>,<NA>,100.0,...,NaN,<NA>,117237.0,118433.0,3684.0,3781.0,26362.0,26136.0,https://www.youtube.com/watch?v=nPdnfKhePSY&t=4s,NaN
7,8,2024-12-27 09:26:15.702,2025-01-15 09:50:21.795,Apple MacBook Pro M4,False,2024,<NA>,<NA>,<NA>,72.4,...,NaN,<NA>,37937.0,NaN,3683.0,NaN,15054.0,NaN,<NA>,NaN
8,9,2024-12-27 09:35:37.480,2025-04-11 10:38:35.471,Asus Vivobook S 14,False,2024,Tiêu thụ trung bình 36W điện,<NA>,<NA>,75.0,...,NaN,<NA>,28764.0,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
9,10,2024-12-28 21:29:35.471,2025-01-12 15:26:05.905,MSI Stealth 18 Mercedes AMG,True,2024,Tiêu thụ trung bình 110W,<NA>,<NA>,99.0,...,97.0,<NA>,156177.0,NaN,NaN,NaN,NaN,NaN,<NA>,NaN


## Thử nghiệm Mô hình hóa DWH (Medallion & Feature Engineering) theo `DWH_plan.md`

In [10]:
# THỬ NGHIỆM Ý TƯỞNG THIẾT KẾ DWH (MEDALLION & FEATURE ENGINEERING)
# File này mô phỏng các công thức tính toán Feature Engineering cho Domain 1 và Domain 2 trước khi đưa vào dbt.

import numpy as np
import pandas as pd

print("--- THỬ NGHIỆM DOMAIN 1: HARDWARE & PERFORMANCE (OBT) ---")

# Truy vấn dữ liệu thực nghiệm từ ClickHouse
obt_test_query = f"""
WITH latest_benchmark AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY laptop_model_id ORDER BY changed_on DESC, id DESC) as rn
        FROM `{TARGET_DB}`.`laptop_benchmark_result`
        WHERE is_active = 1
    )
    WHERE rn = 1
)
SELECT 
    lm.id AS laptop_id,
    lm.name AS laptop_name,
    lm.battery_capacity_whr,
    lm.laptop_weight,
    lm.cpu_note,
    lm.gpu_note,
    
    # Raw benchmark fields
    bench.office_battery_result_minutes,
    bench.geekbench_6_cpu_single_core_plugged_in,
    bench.geekbench_6_cpu_single_core_battery,
    bench.geekbench_6_cpu_multi_core_plugged_in,
    bench.geekbench_6_cpu_multi_core_battery,
    bench.geekbench_6_compute_gpu_plugged_in,
    bench.geekbench_6_compute_gpu_battery
FROM `{TARGET_DB}`.`laptop_model` lm
LEFT JOIN latest_benchmark bench ON lm.id = bench.laptop_model_id
WHERE lm.is_active = 1
LIMIT 5
"""

df_obt = client.query_df(obt_test_query)

# 1. Feature Engineering: Tỷ lệ suy hao hiệu năng khi dùng pin (%)
df_obt["cpu_single_battery_retention_pct"] = (df_obt["geekbench_6_cpu_single_core_battery"] / df_obt["geekbench_6_cpu_single_core_plugged_in"]) * 100
df_obt["cpu_multi_battery_retention_pct"] = (df_obt["geekbench_6_cpu_multi_core_battery"] / df_obt["geekbench_6_cpu_multi_core_plugged_in"]) * 100
df_obt["gpu_compute_battery_retention_pct"] = (df_obt["geekbench_6_compute_gpu_battery"] / df_obt["geekbench_6_compute_gpu_plugged_in"]) * 100

# 2. Feature Engineering: Mobility Score (Chỉ số di động)
df_obt["mobility_score"] = df_obt["office_battery_result_minutes"] / (df_obt["laptop_weight"] / 1000.0)

# 3. Feature Engineering: Semantic Text Chunk (Gộp văn bản phục vụ RAG)
# Viết hàm helper tường minh để tránh lỗi NAType boolean ambiguity
def build_semantic_chunk(row):
    name = row["laptop_name"]
    weight_val = row["laptop_weight"]
    battery_val = row["battery_capacity_whr"]
    cpu_n = row["cpu_note"]
    gpu_n = row["gpu_note"]
    
    # Kiểm tra rỗng an toàn cho kiểu dữ liệu Nullable Pandas
    weight_str = f"{float(weight_val)/1000.0:.2f} kg" if pd.notna(weight_val) else "N/A"
    battery_str = f"{float(battery_val):.1f} Wh" if pd.notna(battery_val) else "N/A"
    cpu_str = str(cpu_n) if pd.notna(cpu_n) and cpu_n else "Không có"
    gpu_str = str(gpu_n) if pd.notna(gpu_n) and gpu_n else "Không có"
    
    return f"Laptop Model: {name}. Trọng lượng: {weight_str}. Dung lượng pin: {battery_str}. Ghi chú CPU: {cpu_str}. Ghi chú GPU: {gpu_str}."

df_obt["semantic_text_chunk"] = df_obt.apply(build_semantic_chunk, axis=1)

display(df_obt[["laptop_name", "cpu_single_battery_retention_pct", "mobility_score", "semantic_text_chunk"]].head())


print("\n--- THỬ NGHIỆM DOMAIN 2: CLICKSTREAM & SEO (EXTRACT JSON) ---")

tracking_test_query = f"""
SELECT 
    id,
    event_name,
    event_data,
    device,
    event_local_timestamp
FROM `{TARGET_DB}`.`user_event_tracking`
LIMIT 3
"""
df_track = client.query_df(tracking_test_query)

import json

def safe_extract_json(json_str, key):
    try:
        data = json.loads(json_str)
        val = data.get(key, None)
        return val if pd.notna(val) else None
    except:
        return None

df_track["extracted_page_name"] = df_track["event_data"].apply(lambda x: safe_extract_json(x, "page_name"))
df_track["extracted_os_name"] = df_track["device"].apply(lambda x: safe_extract_json(x, "os_name"))
df_track["extracted_device_type"] = df_track["device"].apply(lambda x: safe_extract_json(x, "device_type"))

display(df_track[["id", "event_name", "extracted_page_name", "extracted_os_name", "extracted_device_type"]])


--- THỬ NGHIỆM DOMAIN 1: HARDWARE & PERFORMANCE (OBT) ---


,laptop_name,cpu_single_battery_retention_pct,mobility_score,semantic_text_chunk
0,Asus Zenbook S 14 OLED,64.020581,777.500000,Laptop Model: Asus Zenbook S 14 OLED. Trọng lượng: 1.20 kg. Dung lượng pin: 72.0 Wh. Ghi chú CPU: Không có. Ghi chú GPU: Không có.
1,Expertbook P5,NaN,621.749409,Laptop Model: Expertbook P5. Trọng lượng: 1.27 kg. Dung lượng pin: 63.0 Wh. Ghi chú CPU: Không có. Ghi chú GPU: Không có.
2,Dell XPS 13 2024,NaN,691.803279,Laptop Model: Dell XPS 13 2024. Trọng lượng: 1.22 kg. Dung lượng pin: 55.0 Wh. Ghi chú CPU: Không có. Ghi chú GPU: Không có.
3,HP OMEN Transcend 14,NaN,206.707317,Laptop Model: HP OMEN Transcend 14. Trọng lượng: 1.64 kg. Dung lượng pin: 71.0 Wh. Ghi chú CPU: Không có. Ghi chú GPU: Không có.
4,Asus TUF Gaming A14 (2024),100.573271,222.142857,Laptop Model: Asus TUF Gaming A14 (2024). Trọng lượng: 1.40 kg. Dung lượng pin: 72.4 Wh. Ghi chú CPU: Tiêu thụ điện trung bình 80W. Ghi chú GPU: Không có.



--- THỬ NGHIỆM DOMAIN 2: CLICKSTREAM & SEO (EXTRACT JSON) ---


,id,event_name,extracted_page_name,extracted_os_name,extracted_device_type
0,1,pageview,DeviceDetail,Mac OS,None
1,2,pageview,Home,Mac OS,None
2,3,pageview,DeviceDetail,Mac OS,None
